# DSpark 草稿模型训练：Open-PerfectBlend 的 prompt + Target 自己生成的 response

**状态：脚本已写好，尚未运行。** 模型和损失已按论文（arXiv 2607.05147）改写，代码在 `dspark_paper.py` 和 `dspark_loss.py`；旧的 `dflash_big.py` 和 `dflash_big.pt` 不动。

流程：
1. 从 Open-PerfectBlend（1,420,909 条对话）随机抽 **50,000 条 prompt**（只取第一轮 human 的话）+ 500 条验证 prompt。数据集自带的 response 一律不读、不用。
2. 用 `Qwen3.5-0.8B-Base`（Target）对每条 prompt 做**纯温度采样**生成 response（on-policy）。两张 GPU 各跑一个进程，可断点续跑。
3. 训练 `DSparkDrafter`（5 层，约 135.2M）：**Target 全程冻结，只更新 drafter 的参数**。Target 的特征在训练时实时计算，不落盘。
4. 在验证集上评估逐位置接受率和 τ（含 bonus token），并和旧的 `dflash_big.pt` 对比。

**温度**：全流程只有一个 `TEMP`。生成 response、损失里的 p_target 和 q_draft、验证时的接受率都用它。

**损失**（论文 Eq. 9–12）：`L = Σ_k w_k (0.1·CE_k + 0.9·TV_k + 1.0·BCE_k)`，`w_k = exp(-(k-1)/7)`，详见 §3。

In [ ]:
import os, sys, json, time, math, random, glob, hashlib, subprocess, collections
import torch
import torch.nn.functional as F
import pyarrow.parquet as pq

TRAIN_DIR   = '/home/dev/KAP/train'                   # 训练脚本
SRC_DIR     = '/home/dev/KAP/src4dspark'              # DSpark 模型 / 推理源码
PARAM_DIR   = '/home/dev/KAP/parameter/draft'         # draft 权重
TARGET_PATH = '/home/dev/KAP/parameter/target/Qwen3.5-0.8B-Base-original'   # 只读的原始 target 拷贝
PB_DIR      = '/home/dev/KAP/data/open_perfectblend/data'
WORK        = '/home/dev/KAP/data/pb_dspark'          # 本 notebook 的全部产物
os.makedirs(WORK + '/gen', exist_ok=True)
sys.path.insert(0, TRAIN_DIR)
sys.path.insert(0, SRC_DIR)

# ---- 数据 ----
N_TRAIN, N_VAL = 50_000, 500      # 训练 prompt 数 / 验证 prompt 数
SEED = 0
MAX_PROMPT_TOK = 1024             # 套完 chat template 后超过这个长度的 prompt 丢弃
# ---- 采样温度：生成 / 损失 / 验证共用 ----
TEMP = 1.0                        # 论文评测用 T=1.0。Base 模型在 T=1.0 纯采样下文本可能较发散，我没测过；改 TEMP 必须重新生成 response
# ---- 生成（Target 纯温度采样，worker 里写死 top_p=1.0, top_k=0）----
MAX_NEW, GEN_BS = 512, 128
GPUS = [0, 1]
GEN_LIMIT = 256                   # 冒烟测试：每卡只生成这么多条；确认没问题后改成 0 跑全量
MIN_RESP = 16                     # response 少于这么多 token 的样本不参与训练

## 1. 抽取 prompt
按全局行号随机抽样（自然混合比例：其中 MetaMathQA + orca-math 两个纯数学来源合计约 41%，UltraInteract 约 19%），先多抽 25% 作候选，再过滤（第一轮必须是 human、非空、套模板后不超过 1024 token）和精确去重，打乱后截到 50,500 条（前 500 条做验证集）。

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(TARGET_PATH)
EOS = tok.eos_token_id
IM_END = tok.convert_tokens_to_ids('<|im_end|>')

files = sorted(glob.glob(PB_DIR + '/*.parquet'))
sizes = [pq.ParquetFile(f).metadata.num_rows for f in files]
TOTAL = sum(sizes)
print(f'{len(files)} shards, {TOTAL:,} conversations')

rng = random.Random(SEED)
want_all = set(rng.sample(range(TOTAL), int((N_TRAIN + N_VAL) * 1.25)))
cands, off, t0 = [], 0, time.time()
for f, n in zip(files, sizes):
    want = sorted(i - off for i in want_all if off <= i < off + n)
    if want:
        tbl = pq.read_table(f, columns=['conversations', 'source'])
        conv, srcs = tbl.column('conversations'), tbl.column('source')
        for i in want:
            c = conv[i].as_py()
            if c and c[0]['from'] == 'human' and c[0]['value'].strip():   # 只取第一轮 human，不使用 gpt 的回答
                cands.append({'gid': off + i, 'source': srcs[i].as_py(), 'prompt': c[0]['value']})
    off += n
print(f'{len(cands)} candidates, {time.time() - t0:.0f}s')

In [ ]:
def render(p):
    return tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)

rng.shuffle(cands)                 # 候选是按分片顺序收集的，截断前必须打乱
seen, items = set(), []
for c in cands:
    if c['prompt'] in seen:
        continue
    ids = tok(render(c['prompt']), add_special_tokens=False)['input_ids']
    if len(ids) > MAX_PROMPT_TOK:
        continue
    seen.add(c['prompt'])
    c['ids'] = ids
    items.append(c)
    if len(items) == N_TRAIN + N_VAL:
        break
assert len(items) == N_TRAIN + N_VAL, len(items)
for k, c in enumerate(items):
    c['id'] = k
    c['split'] = 'val' if k < N_VAL else 'train'

torch.save(items, WORK + '/prompts.pt')
with open(WORK + '/prompts_raw.jsonl', 'w', encoding='utf-8') as f:
    for c in items:
        f.write(json.dumps({'id': c['id'], 'split': c['split'], 'source': c['source'], 'prompt': c['prompt']}, ensure_ascii=False) + chr(10))

L = sorted(len(c['ids']) for c in items)
n = len(L)
print('prompt tokens (含 chat template) p50/p90/p99/max:', L[n // 2], L[9 * n // 10], L[99 * n // 100], L[-1])
for k, v in collections.Counter(c['source'] for c in items).most_common():
    print(f'{v:6d}  {k}')

## 2. 用 Target 生成 response
- prompt 用 Qwen3.5 的 chat template 渲染（`<|im_start|>user ... <|im_start|>assistant` + 空 think 块），Base 模型直接续写
- **纯温度采样**：`temperature=TEMP, top_p=1.0, top_k=0`，样本严格服从 `softmax(logits / TEMP)`，和损失里的 p_target 是同一个分布。模型目录里没有 `generation_config.json`，`generate` 的默认 top_k 等是否生效不确定，所以 worker 里显式写死，并把实际生效的 generation_config 打到日志里
- 最多 512 个新 token，遇到 `<|im_end|>` 或 `<|endoftext|>` 停止
- 没有 vLLM，用 HF `generate`；按 prompt 长度排序后再切 batch，尽量少左 padding
- 每张 GPU 一个独立进程（按 `id % 2` 分片），每 8 个 batch 落一个 shard，重启后自动跳过已完成的 id
- 吞吐没测过。先用 `GEN_LIMIT=256` 冒烟，看 tok/s 再估全量耗时

In [ ]:
%%writefile /home/dev/KAP/train/pb_gen_worker.py
'''Generate Target responses for one shard of the prompt list (one process per GPU).
Pure temperature sampling (top_p=1.0, top_k=0): samples follow softmax(logits / T) exactly.'''
import argparse, glob, os, time
import torch
from transformers import AutoModelForImageTextToText, AutoTokenizer

ap = argparse.ArgumentParser()
ap.add_argument('--rank', type=int, required=True)
ap.add_argument('--world', type=int, required=True)
ap.add_argument('--work', required=True)
ap.add_argument('--target', required=True)
ap.add_argument('--max-new', type=int, default=512)
ap.add_argument('--temperature', type=float, required=True)
ap.add_argument('--bs', type=int, default=128)
ap.add_argument('--limit', type=int, default=0)
a = ap.parse_args()

tok = AutoTokenizer.from_pretrained(a.target)
EOS = tok.eos_token_id
IM_END = tok.convert_tokens_to_ids('<|im_end|>')
model = AutoModelForImageTextToText.from_pretrained(a.target, dtype=torch.bfloat16, device_map='cuda:0').eval()
print('generation_config:', model.generation_config, flush=True)

items = torch.load(a.work + '/prompts.pt')
mine = [c for c in items if c['id'] % a.world == a.rank]
if a.limit:
    mine = mine[:a.limit]
out_dir = a.work + '/gen'
done = set()
shards = sorted(glob.glob(f'{out_dir}/shard_r{a.rank}_*.pt'))
for f in shards:
    done.update(r['id'] for r in torch.load(f))
mine = [c for c in mine if c['id'] not in done]
mine.sort(key=lambda c: len(c['ids']))
print(f'rank {a.rank}: {len(mine)} prompts to do, {len(done)} already done', flush=True)

buf, shard_k, n_tok, t0 = [], len(shards), 0, time.time()
for b0 in range(0, len(mine), a.bs):
    batch = mine[b0:b0 + a.bs]
    L = max(len(c['ids']) for c in batch)
    ids = torch.full((len(batch), L), EOS, dtype=torch.long)
    am = torch.zeros((len(batch), L), dtype=torch.long)
    for i, c in enumerate(batch):                      # 左 padding
        n = len(c['ids'])
        ids[i, L - n:] = torch.tensor(c['ids'])
        am[i, L - n:] = 1
    with torch.inference_mode():
        gen = model.generate(input_ids=ids.cuda(), attention_mask=am.cuda(), max_new_tokens=a.max_new,
                             do_sample=True, temperature=a.temperature, top_p=1.0, top_k=0,
                             eos_token_id=[IM_END, EOS], pad_token_id=EOS)
    for c, row in zip(batch, gen[:, L:].cpu().tolist()):
        stop = next((j for j, t in enumerate(row) if t in (IM_END, EOS)), None)
        resp = row if stop is None else row[:stop + 1]  # 保留结尾的 im_end / eos
        buf.append({'id': c['id'], 'resp': resp, 'finished': stop is not None})
        n_tok += len(resp)
    last = b0 + a.bs >= len(mine)
    if len(buf) >= a.bs * 8 or last:
        tmp = f'{out_dir}/shard_r{a.rank}_{shard_k:04d}.pt.tmp'
        torch.save(buf, tmp)
        os.replace(tmp, tmp[:-4])                      # 原子落盘，崩溃不会留下半个 shard
        shard_k, buf = shard_k + 1, []
    print(f'rank {a.rank}: {min(b0 + a.bs, len(mine))}/{len(mine)}  {n_tok / (time.time() - t0):.0f} tok/s', flush=True)

### 2.1 启动 / 监控 / 合并
先 `GEN_LIMIT=256` 冒烟：看 tok/s、看解码出来的样例对不对，再把 `GEN_LIMIT` 改成 0 重新启动（已完成的 id 会自动跳过）。

In [ ]:
def launch(limit):
    procs = []
    for r, g in enumerate(GPUS):
        log = open(f'{WORK}/gen/log_r{r}.txt', 'a')
        cmd = [sys.executable, f'{TRAIN_DIR}/pb_gen_worker.py', '--rank', str(r), '--world', str(len(GPUS)),
               '--work', WORK, '--target', TARGET_PATH, '--max-new', str(MAX_NEW),
               '--temperature', str(TEMP), '--bs', str(GEN_BS), '--limit', str(limit)]
        procs.append(subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
                                      env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)}))
    return procs

procs = launch(GEN_LIMIT)      # 后台进程，notebook kernel 重启不会杀掉它们

In [ ]:
def gen_status():
    n = sum(len(torch.load(f)) for f in glob.glob(WORK + '/gen/shard_r*_*.pt'))
    print(f'{n:,} responses on disk;  workers alive: {[p.poll() is None for p in procs]}')
    for r in range(len(GPUS)):
        lines = open(f'{WORK}/gen/log_r{r}.txt').read().splitlines()
        print(lines[-1] if lines else '(no log yet)')

gen_status()

In [ ]:
# 合并 shard -> 训练 / 验证数据（kernel 重启后从这里开始即可）
items = torch.load(WORK + '/prompts.pt')
byid = {c['id']: c for c in items}
resp = {}
for f in sorted(glob.glob(WORK + '/gen/shard_r*_*.pt')):
    for r in torch.load(f):
        resp[r['id']] = r
data = {'train': [], 'val': []}
for i, r in resp.items():
    if len(r['resp']) >= MIN_RESP:
        c = byid[i]
        data[c['split']].append({'id': i, 'source': c['source'], 'prompt_ids': c['ids'],
                                 'resp_ids': r['resp'], 'finished': r['finished']})
print({k: len(v) for k, v in data.items()})

rl = sorted(len(s['resp_ids']) for s in data['train'])
print('response tokens mean/p50/p90:', round(sum(rl) / len(rl), 1), rl[len(rl) // 2], rl[9 * len(rl) // 10],
      ' 自然结束比例:', round(sum(s['finished'] for s in data['train']) / len(rl), 3),
      ' 训练 response token 总数:', sum(rl))
by = collections.defaultdict(list)
for s in data['train']:
    by[s['source']].append(len(s['resp_ids']))
for k, v in by.items():
    print(f'{k:55s} n={len(v):6d}  mean_resp={sum(v) / len(v):6.1f}')
for s in data['train'][:2]:                  # 肉眼看两条样例
    print('-' * 60)
    print(tok.decode(s['prompt_ids'][-60:]))
    print('>>>')
    print(tok.decode(s['resp_ids'][:200]))

In [ ]:
# 可选：左 padding 一致性检查。GDN 线性注意力层对 padding 敏感，所以对比 batch 生成与单条生成（贪心，前 16 个 token）
from transformers import AutoModelForImageTextToText
_m = AutoModelForImageTextToText.from_pretrained(TARGET_PATH, dtype=torch.bfloat16, device_map='cuda:0').eval()
enc = [c['ids'] for c in sorted(items[:64], key=lambda c: len(c['ids']))[::16]]
Lm = max(map(len, enc))
_ids = torch.tensor([[EOS] * (Lm - len(x)) + x for x in enc]).cuda()
_am = torch.tensor([[0] * (Lm - len(x)) + [1] * len(x) for x in enc]).cuda()
kw = dict(max_new_tokens=16, do_sample=False, pad_token_id=EOS)
with torch.inference_mode():
    gb = _m.generate(input_ids=_ids, attention_mask=_am, **kw)[:, Lm:]
    for i, x in enumerate(enc):
        gs = _m.generate(input_ids=torch.tensor([x]).cuda(), **kw)[:, len(x):]
        print(i, 'len', len(x), ' batch == single:', bool((gb[i][:gs.shape[1]] == gs[0]).all()))
del _m
torch.cuda.empty_cache()

## 3. 训练 DSpark drafter

模型在 `dspark_paper.py`，采样 / 损失 / 评测在 `dspark_loss.py`。和论文对照：
- **上下文**：anchor 之前**整条序列**的 5 层 Target 特征，`H_ctx = RMSNorm(W_c [H^(l1);…;H^(l5)])`，注入每个草稿层的 K/V（式 2、3）。同一条序列上的多个 anchor 打包进一次前向，用 mask 保证每个 anchor 只看到自己位置之前的上下文和自己的 block
- K/V 投影在上下文行和草稿行之间共享（式 3）；5 层；block 内 8 行（anchor + 7 个 mask）双向可见
- Markov 头 rank 128（论文 256，因词表更大而降低）；置信度头 `c_k = σ(wᵀ[h_k; W1[x_(k-1)]])` 共享 Markov embedding（式 7），与 backbone 端到端训练（论文 5.1）
- **RoPE**：论文没写位置编码。上下文变长后草稿必须知道距离，我给 q/k 加了 RoPE，这是我的补充
- **冻结**：Target 以及共享的 embedding / lm_head 全部冻结，只更新 backbone、Markov 头和置信度头

### 损失函数
`L = Σ_k w_k · (0.1·CE_k + 0.9·TV_k + 1.0·BCE_k)`，`w_k = exp(-(k-1)/7)`，k = 1..7，不做归一化，对 block 取平均
- `CE_k = -log q_k(x*_k)`：x*_k 是 Target 实际采样出的 token
- `TV_k = ‖q_k − p_k‖₁`：p = softmax(logits_target / TEMP)，q = softmax(logits_draft / TEMP)。`1 − TV/2` 就是投机采样第 k 位的接受概率
- `BCE_k`：置信度头对 `c*_k = 1 − TV_k/2` 的二元交叉熵，c* 只当标签，不反传
- Markov 头训练时的条件 token 是 Target 采样出的真实 token（论文没写，沿用此前做法）

### 训练数据怎么取
每步取 16 条序列（prompt + response）过一次 Target，拿 5 层特征和最后一层 hidden；每条序列在 response 内随机取 8 个 anchor，共 128 个 block。不预先缓存特征：5 层 × 1024 × bf16 ≈ 10 KB/token，按每条约 400 token 估算，5 万条约 200 GB，磁盘放不下。

In [ ]:
from transformers import AutoModelForImageTextToText
from dspark_paper import DSparkDrafter, save, load
from dspark_loss import make_blocks, dspark_loss, summarize, evaluate, exp_len, ND

DEV = 'cuda:0'
model = AutoModelForImageTextToText.from_pretrained(TARGET_PATH, dtype=torch.bfloat16, device_map=DEV).eval()
for p in model.parameters():
    p.requires_grad_(False)

WEIGHT_FILE = TARGET_PATH + '/model.safetensors-00001-of-00001.safetensors'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(1 << 26), b''):
            h.update(blk)
    return h.hexdigest()

def fingerprint(m):
    return [(n, float(p.float().sum()), float(p.float().abs().max())) for n, p in m.named_parameters()]

SHA_BEFORE, FP_BEFORE = sha256(WEIGHT_FILE), fingerprint(model)
print('target sha256:', SHA_BEFORE)

In [ ]:
d = DSparkDrafter(model, n_draft=ND, layers=5, markov_rank=128, device=DEV, dtype=torch.float32)
INIT_FROM = None                  # 想热启动就填 checkpoint 路径；默认从随机初始化训练
if INIT_FROM:
    load(d, INIT_FROM)
print(f'drafter {d.n_params / 1e6:.2f}M params = {100 * d.n_params / 873.4e6:.1f}% of target')
_t = {id(p) for p in model.parameters()}
assert not [n for n, p in d.named_parameters() if id(p) in _t], 'target 参数混进了 drafter'

In [ ]:
# 自检（先跑完 §2 的合并 cell，有 data 才能跑）：hidden_states[-1] 过 lm_head 是否等于 Target logits；
# 变长 batch（右 padding）的特征是否和单条一致；block 形状；初始损失
_g = torch.Generator().manual_seed(0)
with torch.no_grad():
    s0 = data['train'][0]
    full0 = s0['prompt_ids'] + s0['resp_ids']
    _o = model(input_ids=torch.tensor([full0], device=DEV), use_cache=False, output_hidden_states=True, logits_to_keep=1)
    diff = (_o.logits[0, -1].float() - model.lm_head(_o.hidden_states[-1][0, -1]).float()).abs().max().item()
    print('logits shape', tuple(_o.logits.shape), ' max|diff| (hidden_states[-1] -> lm_head vs logits):', diff)
    feats, toks, o, p_t = make_blocks(model, EOS, data['train'][:4], 8, _g, TEMP, DEV)
    print('feats', tuple(feats.shape), ' toks', tuple(toks.shape), ' o', tuple(o.shape), ' p_t', tuple(p_t.shape))
    print('p_t 行和:', round(p_t.sum(-1).mean().item(), 4))
    f1 = make_blocks(model, EOS, data['train'][:1], 8, torch.Generator().manual_seed(0), TEMP, DEV)[0]
    print('padded batch vs single sequence, feature max|diff|:', (feats[0, :len(full0)].float() - f1[0, :len(full0)].float()).abs().max().item())
    print('Target argmax 与采样 token 的一致率（仅作参考，应明显高于随机）:', round((p_t.argmax(-1) == toks[..., 1:]).float().mean().item(), 3))
    loss0, st0 = dspark_loss(d, feats, toks, o, p_t, TEMP)
    print('初始损失', round(loss0.item(), 3), {k: round(v, 3) for k, v in summarize(st0).items()})

In [ ]:
EPOCHS, SEQS_PER_STEP, ANCHORS = 10, 16, 8      # 每步 16 条序列 x 8 个 anchor = 128 个 block；论文训练 10 个 epoch
LR, GRAD_CLIP = 1e-3, 1.0                       # 论文没写学习率和优化器，沿用此前的 AdamW + OneCycle
EVAL_EVERY, SAVE_EVERY, LOG_EVERY = 1000, 1000, 100
CKPT, STATE = PARAM_DIR + '/dspark_pb/dflash_pb.pt', PARAM_DIR + '/dspark_pb/train_state.pt'    # 不覆盖旧的 dflash_big.pt
RESUME = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

train_seqs, val_seqs = data['train'], data['val']
TRAIN_STEPS = EPOCHS * (len(train_seqs) // SEQS_PER_STEP)
print('train seqs', len(train_seqs), ' steps', TRAIN_STEPS)

def batch_iter(seqs, bs, rng):
    while True:
        perm = list(range(len(seqs)))
        rng.shuffle(perm)
        for m0 in range(0, len(perm), bs * 32):        # 先攒一大块，块内按长度排序再切 batch，减少 padding
            chunk = sorted(perm[m0:m0 + bs * 32], key=lambda i: len(seqs[i]['prompt_ids']) + len(seqs[i]['resp_ids']))
            bl = [chunk[j:j + bs] for j in range(0, len(chunk), bs)]
            rng.shuffle(bl)
            for b in bl:
                if len(b) == bs:
                    yield [seqs[i] for i in b]

params = [p for p in d.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=TRAIN_STEPS, pct_start=0.03)
step0 = 0
if RESUME and os.path.exists(STATE):
    ck = torch.load(STATE, map_location=DEV, weights_only=False)
    d.load_state_dict(ck['d'])
    opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched'])
    step0 = ck['step']
    print('resumed from step', step0)

In [ ]:
d.train()
rng, gen = random.Random(SEED + step0), torch.Generator().manual_seed(SEED + step0)
it = batch_iter(train_seqs, SEQS_PER_STEP, rng)
w_loss, w_n, w_st = 0.0, 0, None
t0 = time.time()
for step in range(step0 + 1, TRAIN_STEPS + 1):
    feats, toks, o, p_t = make_blocks(model, EOS, next(it), ANCHORS, gen, TEMP, DEV)
    loss, st = dspark_loss(d, feats, toks, o, p_t, TEMP)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
    opt.step()
    sched.step()
    w_loss += loss.item()
    w_n += 1
    w_st = st if w_st is None else {k: w_st[k] + st[k] for k in st}
    if step % LOG_EVERY == 0:
        r = summarize({k: v / w_n for k, v in w_st.items()})
        ce, tv, bce, tau = r['ce'], r['tv'], r['bce'], r['tau']
        ms = (time.time() - t0) / (step - step0) * 1e3
        lr_now = sched.get_last_lr()[0]
        print(f'step {step:6d}  loss {w_loss / w_n:.3f}  CE {ce:.3f}  TV {tv:.3f}  BCE {bce:.3f}  tau {tau:.2f}  lr {lr_now:.2e}  ({ms:.0f} ms/step)')
        w_loss, w_n, w_st = 0.0, 0, None
    if step % EVAL_EVERY == 0:
        v = evaluate(d, model, EOS, val_seqs[:200], TEMP)
        rv = summarize(v)
        ta, tg, vce, vtv = rv['tau'], rv['tau_argmax'], rv['ce'], rv['tv']
        print(f'  [val@{step}] tau {ta:.2f}  tau_argmax {tg:.2f}  CE {vce:.3f}  TV {vtv:.3f}')
        print('    alpha/slot ' + ' '.join(f'{x:.2f}' for x in v['alpha'].tolist()))
        print('    conf/slot  ' + ' '.join(f'{x:.2f}' for x in v['conf'].tolist()))
    if step % SAVE_EVERY == 0 or step == TRAIN_STEPS:
        save(d, CKPT)
        torch.save({'d': d.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(), 'step': step}, STATE)

## 4. 训练后检查
先证明 Target 没被动过；再在验证集（500 条没参与训练的 prompt，response 同样由 Target 按 TEMP 采样生成）上评估，并在同一批 block 上和旧的 `dflash_big.pt` 对比。

指标口径：
- `tau` = `Σ_k Π_(j≤k) α_j + 1`。α_j = Σ min(p, q) 是第 j 位在 T=TEMP 下的接受概率，+1 是 bonus token（论文脚注 4）。这是 teacher-forced 估计，不是端到端实测
- `tau_argmax`：argmax 一致率版本（相当于 T=0），只用来和此前文档里的数字对照
- `conf`：置信度头的平均输出，应接近同位置的 `alpha`
- 旧 drafter 的上下文窗口只有 4 个位置，训练数据也是另一批（Python 续写、T=0.8），对比只说明新旧版本的差别，不是严格的消融

In [ ]:
d.eval()
assert sha256(WEIGHT_FILE) == SHA_BEFORE, 'Target 权重文件被改动'
assert fingerprint(model) == FP_BEFORE, 'Target 参数被改动'
assert all(not p.requires_grad for p in model.parameters())
print('Target：权重文件 sha256 不变；逐参数 sum / absmax 指纹不变；requires_grad 全为 False')

In [ ]:
from dflash_big import DFlashDrafterBig
d_old = DFlashDrafterBig(model, n_draft=ND, layers=4, markov_rank=128, device=DEV, dtype=torch.float32)
load(d_old, PARAM_DIR + '/legacy/dflash_big.pt')
d_old.eval()

@torch.no_grad()
def evaluate_old(seqs, A=8, bs=16, seed=123):
    g = torch.Generator().manual_seed(seed)
    al, hit, n = 0, 0, 0
    for b0 in range(0, len(seqs), bs):
        feats, toks, o, p_t = make_blocks(model, EOS, seqs[b0:b0 + bs], A, g, TEMP, DEV)
        bi = torch.arange(feats.shape[0], device=DEV).repeat_interleave(A)
        win = o.reshape(-1, 1) + torch.arange(-3, 1, device=DEV)                 # 旧版只看 anchor 前 4 个位置
        ctx = feats[bi[:, None], win].float()
        tk = toks.reshape(-1, ND + 1)
        z = d_old.backbone(ctx, d_old.embed(tk[:, 0]).float().unsqueeze(1))
        q = torch.softmax(d_old.logits(z, tk[:, :ND]) / TEMP, -1)
        pt = p_t.reshape(-1, ND, p_t.shape[-1])
        al = al + torch.minimum(pt, q).sum(-1).mean(0)
        hit = hit + (q.argmax(-1) == pt.argmax(-1)).float().mean(0)
        n += 1
    return al / n, hit / n

In [ ]:
def fmt(v):
    return ' '.join(f'{x:.2f}' for x in v.tolist())

v_new = evaluate(d, model, EOS, val_seqs, TEMP)
r = summarize(v_new)
t, ta, ce, tv = r['tau'], r['tau_argmax'], r['ce'], r['tv']
print(f'ALL new (dflash_pb.pt)    tau {t:.2f}  tau_argmax {ta:.2f}  CE {ce:.3f}  TV {tv:.3f}')
print('    alpha/slot ' + fmt(v_new['alpha']) + '    conf/slot ' + fmt(v_new['conf']))
al_o, hit_o = evaluate_old(val_seqs)
print(f'ALL old (dflash_big.pt)   tau {exp_len(al_o) + 1:.2f}  tau_argmax {exp_len(hit_o) + 1:.2f}')
print('    alpha/slot ' + fmt(al_o))
for src in sorted({s['source'] for s in val_seqs}):
    sub = [s for s in val_seqs if s['source'] == src]
    if len(sub) >= 16:
        a_n = evaluate(d, model, EOS, sub, TEMP)['alpha']
        a_o, _ = evaluate_old(sub)
        print(f'{src[-30:]:32s} n={len(sub):3d}  tau new {exp_len(a_n) + 1:.2f}  old {exp_len(a_o) + 1:.2f}')

### 还没做的事
- 端到端：把 `DSparkEngine` 里的 drafter 换成新的，用草稿自己采的 token 做 Markov 条件，实测接受长度。训练时 Markov 头用的是 Target 采样的真实 token，两者的差异要看端到端结果
- STS 校准（论文 §3.2.1）和前缀调度器（§3.2.2）。batch=1 下调度器的收益有限